In [1]:
import pandas as pd

In [2]:
# Load cleaned dataset
df = pd.read_csv("../data/raw/crime_dataset_india.csv")

In [3]:
# Normalize columns
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df["city"] = df["city"].astype(str).str.strip().str.lower()

In [4]:
# Count crimes per city
crime_counts = df.groupby("city").size().reset_index(name="crime_count")

crime_counts.sort_values(by="crime_count", ascending=False).head(10)

,city,crime_count
5,delhi,5400
17,mumbai,4415
2,bangalore,3588
8,hyderabad,2881
13,kolkata,2518
4,chennai,2493
21,pune,2212
1,ahmedabad,1817
10,jaipur,1479
14,lucknow,1456


In [5]:
def assign_hotspot(count):
    if count < 500:
        return "low"
    elif count < 1500:
        return "medium"
    else:
        return "high"

crime_counts["hotspot_level"] = crime_counts["crime_count"].apply(assign_hotspot)

In [6]:
crime_counts.head()

,city,crime_count,hotspot_level
0,agra,764,medium
1,ahmedabad,1817,high
2,bangalore,3588,high
3,bhopal,690,medium
4,chennai,2493,high


In [7]:
CITY_COORDINATES = {
    "delhi": (28.6139, 77.2090),
    "mumbai": (19.0760, 72.8777),
    "bangalore": (12.9716, 77.5946),
    "chennai": (13.0827, 80.2707),
    "kolkata": (22.5726, 88.3639),
    "pune": (18.5204, 73.8567),
    "hyderabad": (17.3850, 78.4867),
    "ahmedabad": (23.0225, 72.5714),
    "jaipur": (26.9124, 75.7873),
    "lucknow": (26.8467, 80.9462),
}

crime_counts["latitude"] = crime_counts["city"].map(
    lambda x: CITY_COORDINATES.get(x, (None, None))[0]
)
crime_counts["longitude"] = crime_counts["city"].map(
    lambda x: CITY_COORDINATES.get(x, (None, None))[1]
)

crime_counts.dropna(inplace=True)

crime_counts.head()


,city,crime_count,hotspot_level,latitude,longitude
1,ahmedabad,1817,high,23.0225,72.5714
2,bangalore,3588,high,12.9716,77.5946
4,chennai,2493,high,13.0827,80.2707
5,delhi,5400,high,28.6139,77.2090
8,hyderabad,2881,high,17.3850,78.4867


In [8]:
crime_counts.shape

(10, 5)

In [9]:
crime_counts.head()

,city,crime_count,hotspot_level,latitude,longitude
1,ahmedabad,1817,high,23.0225,72.5714
2,bangalore,3588,high,12.9716,77.5946
4,chennai,2493,high,13.0827,80.2707
5,delhi,5400,high,28.6139,77.2090
8,hyderabad,2881,high,17.3850,78.4867


# Model Training

In [10]:
# Select ML features
X = crime_counts[["crime_count", "latitude", "longitude"]]

In [13]:
from sklearn.preprocessing import StandardScaler

In [14]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

array([[-0.82304479,  0.39846595, -1.16943313],
       [ 0.62170922, -1.48877013, -0.04517016],
       [-0.2715746 , -1.46790912,  0.55377874],
       [ 2.09991038,  1.44835122, -0.13147288],
       [ 0.04494972, -0.66007541,  0.1544944 ]])

In [15]:
from sklearn.cluster import KMeans

In [16]:
# Train KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
crime_counts["cluster"] = kmeans.fit_predict(X_scaled)

crime_counts[["city", "crime_count", "cluster"]]

,city,crime_count,cluster
1,ahmedabad,1817,1
2,bangalore,3588,2
4,chennai,2493,2
5,delhi,5400,0
8,hyderabad,2881,2
10,jaipur,1479,1
13,kolkata,2518,2
14,lucknow,1456,1
17,mumbai,4415,1
21,pune,2212,1


In [17]:
# Analyze clusters
cluster_summary = crime_counts.groupby("cluster")["crime_count"].mean()
cluster_summary

cluster
0    5400.0
1    2275.8
2    2870.0
Name: crime_count, dtype: float64

In [18]:
# Map cluster to hotspot severity
cluster_map = {
    cluster_summary.idxmin(): "low",
    cluster_summary.idxmax(): "high"
}

In [19]:
# Remaining cluster = medium
remaining = set(cluster_summary.index) - set(cluster_map.keys())
cluster_map[list(remaining)[0]] = "medium"

crime_counts["ai_hotspot_level"] = crime_counts["cluster"].map(cluster_map)

crime_counts[["city", "crime_count", "ai_hotspot_level"]]

,city,crime_count,ai_hotspot_level
1,ahmedabad,1817,low
2,bangalore,3588,medium
4,chennai,2493,medium
5,delhi,5400,high
8,hyderabad,2881,medium
10,jaipur,1479,low
13,kolkata,2518,medium
14,lucknow,1456,low
17,mumbai,4415,low
21,pune,2212,low


In [20]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(kmeans, "../models/hotspot_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

print("Model and scaler saved successfully")


Model and scaler saved successfully


In [21]:
crime_counts[["city", "crime_count", "ai_hotspot_level"]]


,city,crime_count,ai_hotspot_level
1,ahmedabad,1817,low
2,bangalore,3588,medium
4,chennai,2493,medium
5,delhi,5400,high
8,hyderabad,2881,medium
10,jaipur,1479,low
13,kolkata,2518,medium
14,lucknow,1456,low
17,mumbai,4415,low
21,pune,2212,low


In [22]:
import os

# Ensure outputs folder exists
os.makedirs("../outputs", exist_ok=True)

# Save final AI results
crime_counts.to_csv("../outputs/crime_counts_with_ai_labels.csv", index=False)

print("crime_counts_with_ai_labels.csv saved successfully")


crime_counts_with_ai_labels.csv saved successfully
